To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

**Read our [Gemma 3 blog](https://unsloth.ai/blog/gemma3) for what's new in Unsloth and our [Reasoning blog](https://unsloth.ai/blog/r1-reasoning) on how to train reasoning models.**

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft "trl==0.15.2" triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

### Unsloth

Load up `Qwen 2.5 3B Instruct`, and set parameters

In [ ]:
!pip install vllm==0.8.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 MB 34.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: gguf
    Found existing installation: gguf 0.14.0
    Uninstalling gguf-0.14.0:
      Successfully uninstalled gguf-0.14.0
  Attempting uninstall: xformers
    Found existing installation: xformer

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HUGGINGFACE_TOKEN')
login(hf_token)

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
max_seq_length = 512 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.7, # Reduce if out of memory
)


model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-12 08:52:13 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.50.3. vLLM: 0.8.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit with actual GPU utilization = 69.2%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 39.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 512. Num Sequences = 288.
Unsloth: vLLM's KV Cache can use up to 

model-00002-of-00002.safetensors:   0%|          | 0.00/2.16G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

INFO 04-12 08:53:06 [weight_utils.py:281] Time spent downloading weights for unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit: 24.139161 seconds


model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Unsloth 2025.3.19 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


### Data Prep
<a name="Data"></a>

We directly leverage [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) for data prep and all reward functions. You are free to create your own!

In [ ]:
import re
from datasets import load_dataset, Dataset

# Load and prep dataset
SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

# uncomment middle messages for 1-shot prompting
def get_gsm8k_questions(split = "train") -> Dataset:
    data = load_dataset('openai/gsm8k', 'main')[split] # type: ignore
    data = data.map(lambda x: { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']}
        ],
        'answer': extract_hash_answer(x['answer'])
    }) # type: ignore
    return data # type: ignore

dataset = get_gsm8k_questions()

# Reward functions
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}")
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def int_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

README.md:   0%|          | 0.00/7.94k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

# **自定义数据集 prompt设计+reward设计**

In [ ]:
from datasets import load_dataset,Dataset
import json
import re

In [ ]:
SYSTEM_PROMPT = """
你是一个儿童故事分析专家。请从文本中抽取出事件五元组，严格按以下格式输出：(触发词；主语；宾语；时间状语；地点状语)。一个句子中有可能有多个事件，注意辨别，输出时用换行符分隔。
先分析句子结构，识别核心谓语动词（触发词），再匹配施事（主语）、受事（宾语），最后提取时空成分。
注意：每个字段用中文分号分隔，如无内容填"无"，多个主语或宾语用中文逗号分隔。
示例：小男孩一不小心从树上掉了下来. → (掉；小男孩；无；无；从树上)
"""

EXAMPLE_PROMPT = [
    {
        "role": "user",
        "content": "他到石头上面喊青蛙,你在哪里?"
    },
    {
        "role": "assistant",
        "content": "(到；他；石头上面；无；无)\n(喊；他；青蛙,你在哪里；无；无)"
    }
]


In [ ]:
def load_dataset(jsonl_path):
    data = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            # 清洗并验证格式
            # cleaned_event = clean_event_string(item["event"])
            # assert re.match(r"\([^；]+；[^；]+；[^；]+；[^；]+；[^；]+\)", item["event"]), f"格式错误: {item['event']}"
            data.append({
                "text": item["text"],
                "event": item["event"]
            })
    return data

In [ ]:
dataset = load_dataset("/content/output_train_merge.jsonl")
train_data = [{
    "prompt": [
        {"role": "system", "content": SYSTEM_PROMPT},
        *EXAMPLE_PROMPT,
        {"role": "user", "content": item["text"]}
    ],
    "answer": item["event"]
} for item in dataset]

In [ ]:
print(train_data[0])

{'prompt': [{'role': 'system', 'content': '\n你是一个儿童故事分析专家。请从文本中抽取出事件五元组，严格按以下格式输出：(触发词；主语；宾语；时间状语；地点状语)。一个句子中有可能有多个事件，注意辨别，输出时用换行符分隔。\n先分析句子结构，识别核心谓语动词（触发词），再匹配施事（主语）、受事（宾语），最后提取时空成分。\n注意：每个字段用中文分号分隔，如无内容填"无"，多个主语或宾语用中文逗号分隔。\n示例：小男孩一不小心从树上掉了下来. → (掉；小男孩；无；无；从树上)\n'}, {'role': 'user', 'content': '他到石头上面喊青蛙,你在哪里?'}, {'role': 'assistant', 'content': '(到；他；石头上面；无；无)\n(喊；他；青蛙,你在哪里；无；无)'}, {'role': 'user', 'content': '青蛙在瓶子里的时候.'}], 'answer': '(在；青蛙；瓶子里；无；无)'}


In [ ]:
dataset_train_mini=train_data[:1000]

In [ ]:
print(type(dataset_train_mini))

<class 'list'>


In [ ]:
dataset_mini=Dataset.from_list(dataset_train_mini)

In [ ]:
print(dataset_mini[27])

{'prompt': [{'content': '\n你是一个儿童故事分析专家。请从文本中抽取出事件五元组，严格按以下格式输出：(触发词；主语；宾语；时间状语；地点状语)。一个句子中有可能有多个事件，注意辨别，输出时用换行符分隔。\n先分析句子结构，识别核心谓语动词（触发词），再匹配施事（主语）、受事（宾语），最后提取时空成分。\n注意：每个字段用中文分号分隔，如无内容填"无"，多个主语或宾语用中文逗号分隔。\n示例：小男孩一不小心从树上掉了下来. → (掉；小男孩；无；无；从树上)\n', 'role': 'system'}, {'content': '他到石头上面喊青蛙,你在哪里?', 'role': 'user'}, {'content': '(到；他；石头上面；无；无)\n(喊；他；青蛙,你在哪里；无；无)', 'role': 'assistant'}, {'content': '掉下来把罐子给摔碎.', 'role': 'user'}], 'answer': '(掉；无；罐子；无；无)\n(摔碎；无；无；无；无)'}


In [ ]:
def format_reward_func(completions, **kwargs) -> list[float]:
    """检查括号和分号数量"""
    rewards = []
    pattern = r"^\([^；]+；[^；]+；[^；]+；[^；]+；[^；]+\)$"

    for completion in completions:
        content = completion[0]["content"].strip()
        if re.fullmatch(pattern, content):
            rewards.append(0.0)
        else:
            # 根据错误类型细化惩罚
            if "(" not in content or ")" not in content:
                rewards.append(-0.5)  # 括号缺失
            elif len(content.split("；")) != 5:
                rewards.append(-0.3)  # 分号数量错误
            else:
                rewards.append(-0.2)  # 其他格式问题
    return rewards


In [ ]:
!pip install thefuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.4 MB/s eta 0:00:00


In [ ]:
from rapidfuzz import fuzz, utils

In [ ]:
def field_accuracy_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    rewards = []
    for completion, gold_answer in zip(completions, answer):
        pred = completion[0]["content"].strip()
        gold = gold_answer.strip()

        # 分割字段
        pred_fields = pred.strip("()").split("；")
        gold_fields = gold.strip("()").split("；")

        field_scores = []
        for pred_val, gold_val in zip(pred_fields, gold_fields):
            if gold_val == "无":
                if pred_val == "无":
                    score = 0.05  # 抑制"无-无"过度奖励
                else:
                    score = -0.3  # 虚检惩罚
            else:
                if pred_val == "无":
                    score = -0.5  # 漏检惩罚
                else:
                    # 模糊匹配相似度
                    similarity = fuzz.ratio(pred_val, gold_val) / 100
                    score = max(0.0, similarity - 0.3)  # 相似度低于20%视为错误

            field_scores.append(score)

        # 加权求和（触发词权重最高）
        weights = [0.3, 0.25, 0.25, 0.1, 0.1]
        total = sum(s * w for s, w in zip(field_scores, weights))
        rewards.append(total)

    return rewards


In [ ]:
def logic_consistency_reward_func(prompts, completions, **kwargs) -> list[float]:
    """检查触发词与宾语的合理性"""
    rewards = []
    for prompt, completion in zip(prompts, completions):
        text = prompt[-1]["content"]
        pred = completion[0]["content"].strip()

        try:
            trigger = pred.split("；")[0].strip("()")
            object_ = pred.split("；")[2].strip()

            # 简单规则：宾语应出现在原文中
            if object_ != "无" and object_ not in text:
                rewards.append(-0.3)
            else:
                rewards.append(0.1)  # 基础奖励
        except:
            rewards.append(0.0)
    return rewards

In [ ]:
def combined_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    # print("Prompt:", prompts[0][-1]["content"])  # 应输出用户问题文本
    # print("Completion:", completions[0][0]["content"])  # 模型生成的响应
    # print("Answer:", answer[0])  # 真实标签
    format_rewards = format_reward_func(completions)
    field_rewards = field_accuracy_reward_func(prompts, completions, answer)
    logic_rewards = logic_consistency_reward_func(prompts, completions)

    total_rewards = [
        0.1*format + 0.8*field + 0.1*logic
        for format, field, logic in zip(format_rewards, field_rewards, logic_rewards)
    ]
    return [min(max(r, -1), 1) for r in total_rewards]  # 限制在[-1,1]

# **合并行的奖励函数设计**

In [ ]:
LINE_PATTERN = re.compile(r"^\([^；]+；[^；]+；[^；]+；[^；]+；[^；]+\)$")
from typing import List, Dict, Any, Union, Set
from thefuzz import fuzz

In [ ]:
def format_reward_func(completions, **kwargs) -> list[float]:
    """检查每行事件的括号和分号数量"""
    rewards = []

    for completion in completions:
        content = completion[0]["content"].strip()
        lines = [line.strip() for line in content.split("\n") if line.strip()]  # 分割多行事件

        valid_count = sum(1 for line in lines if LINE_PATTERN.fullmatch(line))
        total_lines = len(lines) or 1
        invalid_count = total_lines - valid_count

        line_reward = (valid_count / total_lines) * 0.3
        line_penalty = (invalid_count / total_lines) * -0.2

        rewards.append(line_reward + line_penalty)

    return rewards

In [ ]:
def field_accuracy_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    # print("Prompt:", prompts[0][-1]["content"])  # 应输出用户问题文本
    # print("Answer:", answer[0])  # 真实标签
    # print("Completion:", completions[0][0]["content"])  # 模型生成的响应

    rewards = []
    for completion, gold_answer in zip(completions, answer):
        pred_lines = [line.strip() for line in completion[0]["content"].strip().split("\n") if line.strip()]
        gold_lines = [line.strip() for line in gold_answer.strip().split("\n") if line.strip()]

        # 解析事件字段（容错处理）
        pred_events = [parse_event(line) for line in pred_lines]  # parse_event函数见下文
        gold_events = [parse_event(line) for line in gold_lines]

        pred_events = [event for event in pred_events if event]
        gold_events = [event for event in gold_events if event]

        # 计算事件级F1（考虑顺序）
        tp, fp, fn = 0, 0, 0
        matched_gold: Set[int] = set()

        # 为每个预测事件找最佳匹配
        for p_idx, p_event in enumerate(pred_events):
            best_sim, best_g_idx = -1, -1
            for g_idx, g_event in enumerate(gold_events):
                if g_idx in matched_gold:
                    continue
                # 计算事件相似度（加权字段）
                sim = event_similarity(p_event, g_event)
                if sim > best_sim:
                    best_sim = sim
                    best_g_idx = g_idx

            if best_sim >= 0.75 and best_g_idx != -1:   # 相似度阈值
                tp +=1
                matched_gold.add(best_g_idx)
            else:
                fp +=1

        # 计算未匹配的真实事件
        fn = len(gold_events) - len(matched_gold)

        # F1计算
        precision = tp / max(tp + fp, 1e-9)
        recall = tp / max(tp + fn, 1e-9)
        f1 = 2 * (precision * recall) / max(precision + recall, 1e-9)

        event_diff = abs(len(pred_events) - len(gold_events))
        f1_with_penalty = max(0, f1 * 0.9 - 0.1 * event_diff)  # 每多/少一个事件扣0.1分

        rewards.append(f1_with_penalty)

    return rewards

def parse_event(line: str) -> dict:
    """解析单行事件，返回字段字典（解析失败返回None）"""
    try:
        line = line.strip()
        if not (line.startswith('(') and line.endswith(')')):
            return None

        parts = line.strip("()").split("；")
        if len(parts) != 5:
            return None

        return {
            "trigger": parts[0].strip(),
            "subject": parts[1].strip(),
            "object": parts[2].strip(),
            "time": parts[3].strip(),
            "location": parts[4].strip()
        }
    except Exception:
        return None

def event_similarity(event1: dict, event2: dict) -> float:
    """加权字段相似度计算"""
    weights = {"trigger":0.4, "subject":0.2, "object":0.2, "time":0.1, "location":0.1}
    total = 0.0
    for field, w in weights.items():
        # 处理"无"的特殊情况
        if event1[field] == "无" and event2[field] == "无":
            total += w * 0.5
        else:
            sim = fuzz.ratio(event1[field], event2[field]) / 100.0
            total += w * sim
    return total

In [ ]:
'''
def event_count_reward_func(completions, answer, **kwargs) -> list[float]:
    """检查多生成或少生成的事件数量，并根据差异给予惩罚"""
    rewards = []

    for completion, gold_answer in zip(completions, answer):
        pred_lines = [line.strip() for line in completion[0]["content"].strip().split("\n") if line.strip()]
        gold_lines = [line.strip() for line in gold_answer.strip().split("\n") if line.strip()]


        # 计算预测事件数量和真实事件数量的差异
        pred_event_count = len(pred_lines)
        gold_event_count = len(gold_lines)

        # Ensure both lists have at least some content to compare
        if not pred_lines and not gold_lines:
            count_penalty = 0.1  # Both empty is correct
        elif not pred_lines or not gold_lines:
            # One empty but not the other
            count_penalty = -0.2 * max(pred_event_count, gold_event_count)
        else:
            # Normal case - both have content
            event_count_diff = abs(pred_event_count - gold_event_count)
            if event_count_diff > 0:
                count_penalty = -0.2 * event_count_diff
            else:
                count_penalty = 0.1

        rewards.append(count_penalty)

    return rewards
'''

In [ ]:
def logic_consistency_reward_func(prompts, completions, **kwargs) -> list[float]:
    """Check if each element comes from the original text"""
    rewards = []

    for prompt, completion in zip(prompts, completions):
        text = prompt[-1]["content"]
        # Normalize the original text for better matching
        normalized_text = text.lower()
        pred = completion[0]["content"].strip().split("\n")  # First split multiple events by \n

        event_rewards = []  # Store rewards for each event

        for p in pred:  # Process each predicted event
            if not p.strip():
                continue  # Skip empty lines

            try:
                # Parse the event structure
                p = p.strip("()")  # Remove parentheses from the event
                parts = p.split("；")

                # Handle potential IndexError if parts doesn't have enough elements
                if len(parts) < 5:
                    event_rewards.append(-0.3)  # Penalize incorrect format
                    continue

                trigger = parts[0].strip()
                subject = [s.strip() for s in parts[1].split("，") if s.strip()]
                object_ = [o.strip() for o in parts[2].split("，") if o.strip()]
                time = parts[3].strip()
                location = parts[4].strip()

                # Initialize reward and counting variables
                reward = 0.0
                match_score = 0
                total_elements = 0

                # Check all elements
                for element_list, element_type in [
                    ([trigger], "trigger"),
                    (subject, "subject"),
                    (object_, "object"),
                    ([time], "time"),
                    ([location], "location")
                ]:
                    for element in element_list:
                        if element != "无":
                            total_elements += 1
                            normalized_element = element.lower()

                            # More accurate text matching with word boundaries where possible
                            if normalized_element in normalized_text:
                                match_score += 1
                            else:
                                # For debugging: print(f"Element '{element}' ({element_type}) not found in text")
                                reward -= 0.2  # Deduct points for each element not in original text

                # If all elements were found or marked as "无", give base reward
                if total_elements > 0 and match_score == total_elements:
                    reward = 0.3  # Base reward if all elements pass verification

                # Alternative graduated reward approach
                # if total_elements > 0:
                #     reward = 0.3 * (match_score / total_elements)

                event_rewards.append(reward)

            except IndexError:
                # Specific error handling for index errors (missing fields)
                event_rewards.append(-0.3)
            except Exception as e:
                # Generic error handling for other parsing failures
                # For debugging: print(f"Parsing failed: {e}")
                event_rewards.append(-0.2)

        # Average the rewards across all events
        if event_rewards:
            rewards.append(sum(event_rewards) / len(event_rewards))
        else:
            rewards.append(0.0)

    return rewards


In [ ]:
def combined_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    # print("Prompt:", prompts[0][-1]["content"])  # 应输出用户问题文本
    # print("Completion:", completions[0][0]["content"])  # 模型生成的响应
    # print("Answer:", answer[0])  # 真实标签
    format_rewards = format_reward_func(completions)
    field_rewards = field_accuracy_reward_func(prompts, completions, answer)
    logic_rewards = logic_consistency_reward_func(prompts, completions)

    total_rewards = [
        0.3*format + 0.5*field + 0.2*logic
        for format, field, logic in zip(format_rewards, field_rewards, logic_rewards)
    ]
    return [min(max(r, -1), 1) for r in total_rewards]  # 限制在[-1,1]

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import wandb
from google.colab import userdata

wnb_token=userdata.get('wandb_token')
wandb.login(key=wnb_token) # import wandb
run = wandb.init(
    project='GRPO-stage1-7B-test0412',
    entity='FeSCN',
    job_type="training",
    settings=wandb.Settings(init_timeout=120),
    anonymous="allow"
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yuxuan0612 (FeSCN) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    use_vllm = True, # use vLLM for fast inference!
    learning_rate = 1e-5,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 10,
    bf16 = is_bfloat16_supported(),
    fp16 = not is_bfloat16_supported(),
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = 256,
    max_completion_length = 128,
    num_train_epochs = 1, # Set to 1 for a full training run
    # max_steps = 250,
    save_steps = 200,
    max_grad_norm = 0.1,
    report_to = "wandb", # Can use Weights & Biases
    output_dir = "/content/drive/MyDrive/GRPO-stage1-7B-outputs",
)

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        combined_reward_func
    ],
    args = training_args,
    train_dataset = train_data,
)

In [ ]:
trainer.train()

[large output cleared]


In [ ]:
wandb.finish()

train/completion_length,▄▃▄▄▄▂▅█▁▄▆▄▆▅▄▄▄▄█▅▅▆▄▂▃▂▅▅▄▇▂▄▇▄▃▄▂▂▆▅
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇██
train/grad_norm,▆▁▁▂▁▁▁▃▁▁▇█▁▅▄▄▅▄▅▄▃▄▄▃▄▂▅▁▁▁▆▄▄▁▁▅▇▆▇▅
train/kl,▁▁▁▁▁▄▃▃▂▂▄▂▃▄▅█▂▆▄▄▂▃▃▂▄▃▃▃▄▄▄▄▂▃▅▄▄▃▅▃
train/learning_rate,▃▄▄▇▇█████▇▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train/loss,▁▁▁▁▁▂▂▃▂▂▄▂▂▃▄▅█▂▆▄▃▄▃▃▃▂▃▅▃▃▃▃▃▃▄▃▄▅▅▃
train/reward,▄▁▁▃▄▄▃▄▆▅▄▃▅▅▅▇▅▆▄▄▄▆▄▆▅▄▆▆▇▆▅▅▅▄▃▃▅▅▂█
train/reward_std,▇▆▃▂▆▅▄▅█▄▄▆▂▇▅▅▃▂█▃▂▃▃▄▄▆▂▄▅▄▆▇▁▄▆█▇▆▁▄
train/rewards/combined_reward_func,▅▃▇▄▁▅▅▇▇▆▆▇▇▆▇█▆▆▇▆▆▆▇██▆▅▅██▅▇▇▇▇▇▇▆▇▆
total_flos,0


<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "第二天早上小狗和小男孩发现瓶子里的青蛙不见了."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 128,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.36it/s, est. speed input: 55.96 toks/s, output: 103.72 toks/s]


'从你描述的情景来看，小狗和小男孩很可能是发生了某种不幸的事情。青蛙被偷走了。这可能是因为他们无法找回，可能是被其他小动物偷走，或者是被主人不小心遗失了。无论如何，青蛙的消失显然给小男孩和他的家人带来了一定的困扰。在这样的情况下，小男孩和他的家人可能会感到非常担心和困惑。'

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Now we load the LoRA and test:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "那个蜜蜂在蜂窝里飞了出来."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 128,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.93it/s, est. speed input: 343.68 toks/s, output: 27.18 toks/s]


'(飞出；蜜蜂；蜂窝；无；无)'

**保存推理结果**

In [ ]:
def run_inference(input_file, output_file, model, tokenizer):
    # 读取eval.jsonl文件
    with open(input_file, 'r', encoding='utf-8') as infile, open(output_file, 'w', encoding='utf-8') as outfile:
        for line_num, line in enumerate(infile, start=1):
            # 解析每一行的JSON数据
            data = json.loads(line)
            text = data.get('text', '')

            # 生成text
            text_input = tokenizer.apply_chat_template([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": text},
            ], tokenize=False, add_generation_prompt=True)

            # 设置采样参数
            sampling_params = SamplingParams(
                temperature=0.8,
                top_p=0.95,
                max_tokens=128,
            )

            # 推理并获得输出
            output = model.fast_generate(
                text_input,
                sampling_params=sampling_params,
                lora_request=model.load_lora("grpo_saved_lora"),
            )[0].outputs[0].text

            # 创建新的输出数据
            result = {
                'id': line_num,
                'input_text': text,
                'output': output
            }

            # 将结果写入到新的jsonl文件中
            outfile.write(json.dumps(result, ensure_ascii=False) + '\n')


In [ ]:
run_inference(input_file="/content/output_eval_merge.jsonl", output_file="/content/drive/MyDrive/two stage results/stage one/GRPO_1epoch_result.jsonl", model=model, tokenizer=tokenizer)

[large output cleared]


Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>


# **保存到huggingface**

In [ ]:
from huggingface_hub import HfApi

# 首先手动创建一个 repo（在Huggingface网页上新建 Model Repo），或者代码创建：
from huggingface_hub import create_repo

repo_name = "Venassa/Qwen2.5-7B-Children_Narrative_Extraction-event_extraction_GRPO"
create_repo(repo_name, exist_ok=True)

# push 上传
model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

adapter_model.safetensors:   0%|          | 0.00/323M [00:00<?, ?B/s]

Saved model to https://huggingface.co/Venassa/Qwen2.5-7B-Children_Narrative_Extraction-event_extraction_GRPO


README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]